<a href="https://colab.research.google.com/github/IvanMattos89/AI-Career-Agent/blob/main/Agentes_de_IA_com_CrewAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalação da biblioteca de [Crewai](https://crewai.com/open-source)

In [ ]:
!pip install -q -U crewai crewai_tools

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


## Instalação da Biblioteca do Gemini


In [ ]:
!pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.


In [ ]:
!pip show crewai

Name: crewai
Version: 1.15.45
Summary: Cutting-edge framework for orchestrating role-playing, autonomous AI agents. By fostering collaborative intelligence, CrewAI empowers agents to work together seamlessly, tackling complex tasks.
Home-page: https://crewai.com
Author: 
Author-email: Joao Moura <joao@crewai.com>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: aiofiles, aiosqlite, appdirs, cel-python, click, crewai-cli, crewai-core, httpx, instructor, json-repair, json5, jsonref, lancedb, mcp, oauthlib, openai, openpyxl, opentelemetry-api, opentelemetry-exporter-otlp-proto-http, opentelemetry-sdk, pdfplumber, portalocker, pydantic, pydantic-settings, pyjwt, python-dotenv, pyyaml, qdrant-client, regex, tokenizers, tomli, tomli-w
Required-by: crewai-tools


In [ ]:
#importar bibliotecas Gemini e Crewai
import os
import asyncio
import nest_asyncio
from google import genai
from google.colab import userdata
from crewai import Agent, Task, Crew, LLM

In [ ]:
#Aplicar o caminho para permitir as chamadas sincronas/assincronas
#dentro do laço(loop) de repetição do Colab
nest_asyncio.apply()


In [ ]:
#configurar a chave Gemini
#os.environ["GOOGLE_GENAI_PROJECT"] = "crewai"
os.environ["Gemini_API_Key"]=userdata.get('Gemini_API_Key') #MESMA API DOS PROJETOS ANTERIORES

In [ ]:
#Lista os modelos disponíveis
client = genai.Client(api_key=os.environ['Gemini_API_Key'])

gemini_llm = LLM(
    model="gemini/gemini-3.6-flash",
    api_key = os.environ['Gemini_API_Key'],
    temperature =0.7,
    max_retries =5, #tentativas de conexão com modelo da API GRATUITA
    retry_delay = 10, #espera de 10s entre tentativas
)

# Criação dos Agentes

In [ ]:
#Agente de Social Media
criador_social = Agent(
    #definir a personalidade
    role='Criador de conteúdo para Redes Sociais com mais de 15 anos de experiência',
    goal='Produzir posts envolventes sobre tecnologia e inovação de mercado, considerando a contemporaniedade',
    backstory=(
        "Especialista em tranformar conceitos técnicos em mensagens acessíveis e cativantes para redes sociais como Instagram, Linkedin e Tiktok."
    ),
    llm=gemini_llm,
    verbose=True,
)

In [ ]:
#Agente Editor de Conteúdo
editor_social = Agent(
    role= "Editor de Conteúdo Digital Sênior",
    goal= "Ajustar o tom, clareza, legibilidade e formato dos posts para maximizar o engajamento e interação",
    backstory=("Profissional com 20 anos de experiência em conteúdo de copywriting e otimização para plataformas como Linkedin, Reddit, Stackoverflow, Instagram e Tiktok"),
    llm=gemini_llm,
    verbose=True,
)

In [ ]:
#Criação da Tarefa do post para criador_social

tarefa_post =Task(
    description=(
        "Crie um post para redes sociais explicando de forma simples o que são agentes autonomos de IA, e como eles podem ser utilizados por profissionais de marketing e criadores de conteúdo. O post deve ser curto, direto, objetivo com tom inspirador."
    ),
    expected_output="Um post com até 280 caracteres, pronto para publicação, com linguagem envolvente e acessível, não faça utilização de termos muito técnicos ",
    agent=criador_social,
)

In [ ]:
#Criação da Tarefa de Edição para o Agent editor_social

tarefa_revisao = Task(
    description=("Revise o conteúdo do post criado sobre agentes autonomos de IA para redes sociais. Ajuste o tom para torná-lo acessível e inspirador, corrija eventuais erros de linguagem e otimize o formato para as plataformas como Linkedin e Instagram."),
    expected_output="Post revisado e otimizado para redes sociais, com tom envolvente e pronto para revisão humana para ser publicado.",
    agent=editor_social,
)


In [ ]:
#criação do time/equipe dos agentes e interação
equipe_social = Crew(
    agents=[criador_social, editor_social],
    tasks=[tarefa_post,tarefa_revisao],
    verbose=True,
)

In [ ]:
async def executar_crew():
  return await equipe_social.kickoff_async()

resultado = asyncio.run(executar_crew())
print(resultado)

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 6566fbce-9cff-4b35-80a1-9cd4f643746e                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: Crie um post para redes sociais explicando de forma simples o que     │
│  são agentes autonomos de I

ERROR:root:Google Gemini API error: 503 - This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.


An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
╭──────────────────────────────── ❌ LLM Error ────────────────────────────────╮
│                                                                              │
│  LLM Call Failed                                                             │
│  Error: Google Gemini API error: 503 - This model is currently experiencing  │
│  high demand. Spikes in demand are usually temporary. Please try again       │
│  later.                                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯


ERROR:crewai.flow.runtime:Error executing listener call_llm_and_parse: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}



An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
╭────────────────────────────── 🤖 Agent Started ──────────────────────────────╮
│                                                                              │
│  Agent: Criador de conteúdo para Redes Sociais com mais de 15 anos de        │
│  experiência                                                                 │
│                                                                              │
│  Task: Crie um post para redes sociais explicando de forma simples o que     │
│  são agentes autonomos de IA, e como eles podem ser utilizados por           │
│  profissionais de marketing e criadores de conteúdo. O post deve ser curto,  │
│  direto, objetivo com tom inspirador.                                        │
│ 

ERROR:root:Google Gemini API error: 503 - This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.


An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
╭──────────────────────────────── ❌ LLM Error ────────────────────────────────╮
│                                                                              │
│  LLM Call Failed                                                             │
│  Error: Google Gemini API error: 503 - This model is currently experiencing  │
│  high demand. Spikes in demand are usually temporary. Please try again       │
│  later.                                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯


ERROR:crewai.flow.runtime:Error executing listener call_llm_and_parse: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}



An unknown error occurred. Please check the details below.
Error details: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
╭────────────────────────────── 📋 Task Failure ───────────────────────────────╮
│                                                                              │
│  Task Failed                                                                 │
│  Name: Crie um post para redes sociais explicando de forma simples o que     │
│  são agentes autonomos de IA, e como eles podem ser utilizados por           │
│  profissionais de marketing e criadores de conteúdo. O post deve ser curto,  │
│  direto, objetivo com tom inspirador.                                        │
│  Agent: Criador de conteúdo para Redes Sociais com mais de 15 anos de        │
│  experiência                                                                 │
│ 

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}